In [1]:
import pandas as pd

In [2]:
equity_df = pd.read_csv('equity_value_data_v2.csv')
equity_df

,timestamp,close_equity,user_id,date,daily_equity_change
0,2016-08-18T00:00:00Z,1211.6055,0012db34aa7b083f5714e7831195e54d,2016-08-18,NaN
1,2016-08-19T00:00:00Z,1173.5640,0012db34aa7b083f5714e7831195e54d,2016-08-19,-38.0415
2,2016-08-22T00:00:00Z,1253.0597,0012db34aa7b083f5714e7831195e54d,2016-08-22,79.4957
3,2016-08-23T00:00:00Z,1252.9050,0012db34aa7b083f5714e7831195e54d,2016-08-23,-0.1547
4,2016-08-24T00:00:00Z,1262.1360,0012db34aa7b083f5714e7831195e54d,2016-08-24,9.2310
...,...,...,...,...,...
1119153,2017-08-14T00:00:00Z,2156.2400,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-14,47.9100
1119154,2017-08-15T00:00:00Z,2134.7100,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-15,-21.5300
1119155,2017-08-16T00:00:00Z,2152.1200,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-16,17.4100
1119156,2017-08-17T00:00:00Z,2042.2800,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-17,-109.8400


In [3]:
# Function to create full date range for one user
def complete_user_dates(user_df):
    full_dates = pd.DataFrame({'date': pd.date_range(user_df['date'].min(), user_df['date'].max())})
    full_dates['user_id'] = user_df['user_id'].iloc[0]
    return full_dates

In [4]:
complete_dates_df = (
    equity_df.groupby('user_id')
    .apply(complete_user_dates)
    .reset_index(drop=True)
)

equity_df['date'] = pd.to_datetime(equity_df['date'])
complete_dates_df['date'] = pd.to_datetime(complete_dates_df['date'])


/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_39202/626639024.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(complete_user_dates)


In [5]:
# Merge to bring in actual trades
full_equity_df = complete_dates_df.merge(
    equity_df[['user_id', 'date', 'close_equity', 'timestamp']],
    on=['user_id', 'date'],
    how='left'
)

# Add binary flag: 1 = traded, 0 = no trade
full_equity_df['active_day'] = full_equity_df['timestamp'].notna().astype(int)

In [6]:
# Forward fill close_equity for inactive days
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])
full_equity_df['close_equity'] = full_equity_df.groupby('user_id')['close_equity'].ffill()

In [7]:
# Drop timestamp as it no longer has useful information after date was split off.
full_equity_df.drop('timestamp', axis=1, inplace=True)

In [8]:
# Re-add daily change
full_equity_df['daily_equity_change'] = full_equity_df.groupby('user_id')['close_equity'].diff()

In [9]:
full_equity_df

,date,user_id,close_equity,active_day,daily_equity_change
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,1,NaN
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,1,-38.0415
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,1173.5640,0,0.0000
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,1173.5640,0,0.0000
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,1,79.4957
...,...,...,...,...,...
1647990,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,1,47.9100
1647991,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,1,-21.5300
1647992,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,1,17.4100
1647993,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,1,-109.8400


# NOTE: The addition of non-active days to equity_df will not affect the aggregates in features_df, as those were taken from the old version.

Below it seems to be showing that no user ever dropped under $10 equity.

In [14]:
full_equity_df.nsmallest(10, 'close_equity')

,date,user_id,close_equity,active_day,daily_equity_change
43005,2017-02-03,071aeed58c5fdb91c6b333e9b76b8400,10.0,1,NaN
43006,2017-02-04,071aeed58c5fdb91c6b333e9b76b8400,10.0,0,0.0
43007,2017-02-05,071aeed58c5fdb91c6b333e9b76b8400,10.0,0,0.0
43008,2017-02-06,071aeed58c5fdb91c6b333e9b76b8400,10.0,0,0.0
43009,2017-02-07,071aeed58c5fdb91c6b333e9b76b8400,10.0,0,0.0
46419,2016-08-17,07ee5bf682719b00115e92fcf4fddb8f,10.0,1,NaN
46420,2016-08-18,07ee5bf682719b00115e92fcf4fddb8f,10.0,1,0.0
46421,2016-08-19,07ee5bf682719b00115e92fcf4fddb8f,10.0,1,0.0
46422,2016-08-20,07ee5bf682719b00115e92fcf4fddb8f,10.0,0,0.0
46423,2016-08-21,07ee5bf682719b00115e92fcf4fddb8f,10.0,0,0.0


Clearly I have missed something, or the dataset is flawed, or this was a trick question.

In [10]:
features_df = pd.read_csv('features_data_v2.csv')
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,first_trade,last_trade,trading_span,total_trades,time_per_trade,trades_per_day,max_change,min_change,max_abs_change,mean_change,median_change
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034,2016-11-16,2017-07-12,238,190,0.174365,1.252632,42.4800,-5.5600,42.4800,0.225397,-0.0400
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638,2016-11-16,2017-07-12,238,252,0.065768,0.944444,9.7100,-9.8100,9.8100,0.239761,0.5700
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f,2016-11-16,2017-07-12,238,252,0.039716,0.944444,0.0000,0.0000,0.0000,0.000000,0.0000
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393,2016-11-16,2017-07-12,238,143,0.007214,1.664336,107.6900,-17.6000,107.6900,0.621479,-0.0500
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91,2016-11-16,2017-07-12,238,252,0.032489,0.944444,138.5644,-139.2006,139.2006,0.413386,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974,2016-08-16,2017-08-15,364,252,0.033092,1.444444,290.8500,-65.9900,290.8500,2.850916,0.8300
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6,2016-08-16,2017-08-15,364,99,0.073145,3.676768,195.1620,-515.1800,515.1800,3.064490,0.1750
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135,2016-08-16,2017-08-15,364,197,0.116585,1.847716,209.3228,-15.5036,209.3228,2.096224,0.0068
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a,2016-08-16,2017-08-15,364,141,0.073322,2.581560,195.3400,-40.8700,195.3400,1.615500,-0.3250


In [12]:
# Create a boolean Series per user: True if close_equity >= 10 ever
high_equity_flag = equity_df.groupby('user_id')['close_equity'].apply(lambda x: (x >= 10).any()).astype(int)

# Create a boolean Series per user: True if close_equity < 10 ever
low_equity_flag = equity_df.groupby('user_id')['close_equity'].apply(lambda x: (x < 10).any()).astype(int)


# Merge / map into features_df
features_df['ever_10_or_above'] = features_df['user_id'].map(high_equity_flag)
features_df['ever_below_10'] = features_df['user_id'].map(low_equity_flag)

features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,first_trade,...,total_trades,time_per_trade,trades_per_day,max_change,min_change,max_abs_change,mean_change,median_change,ever_10_or_above,ever_below_10
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034,2016-11-16,...,190,0.174365,1.252632,42.4800,-5.5600,42.4800,0.225397,-0.0400,1,0
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638,2016-11-16,...,252,0.065768,0.944444,9.7100,-9.8100,9.8100,0.239761,0.5700,1,0
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f,2016-11-16,...,252,0.039716,0.944444,0.0000,0.0000,0.0000,0.000000,0.0000,1,0
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393,2016-11-16,...,143,0.007214,1.664336,107.6900,-17.6000,107.6900,0.621479,-0.0500,1,0
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91,2016-11-16,...,252,0.032489,0.944444,138.5644,-139.2006,139.2006,0.413386,0.0000,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974,2016-08-16,...,252,0.033092,1.444444,290.8500,-65.9900,290.8500,2.850916,0.8300,1,0
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6,2016-08-16,...,99,0.073145,3.676768,195.1620,-515.1800,515.1800,3.064490,0.1750,1,0
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135,2016-08-16,...,197,0.116585,1.847716,209.3228,-15.5036,209.3228,2.096224,0.0068,1,0
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a,2016-08-16,...,141,0.073322,2.581560,195.3400,-40.8700,195.3400,1.615500,-0.3250,1,0


In [13]:
features_df['ever_below_10'].value_counts()

ever_below_10
0    5584
Name: count, dtype: int64

Every user reached $10 equity at least once.